In [53]:
import pymupdf
import json
import os
import re
import collections
from typing import List, Dict, Tuple
from tqdm import tqdm



In [54]:

pdf_file = "Grammar_books/kalamang.pdf"

print("Opening PDF file...")
doc = pymupdf.open(pdf_file)
print("PDF file opened successfully.")
print("Number of pages in PDF file:", doc.page_count)

Opening PDF file...
PDF file opened successfully.
Number of pages in PDF file: 573


In [55]:
toc = doc.get_toc()

In [56]:
toc

[[1, 'Acknowledgments', 13],
 [1, 'Abbreviations', 15],
 [1, 'Ringkasan Bahasa Indonesia [Indonesian summary]', 19],
 [1, '1 Introduction', 25],
 [2, '1.1 Local setting', 25],
 [3, '1.1.1 Physical geography', 25],
 [3, '1.1.2 Linguistic geography', 28],
 [2, '1.2 History of settlement and contact', 29],
 [2, '1.3 Ethnographic and socio-economic remarks', 31],
 [2, '1.4 Sociolinguistic situation', 37],
 [2,
  '1.5 Previous accounts of Kalamang and its genealogical affiliations',
  41],
 [2, '1.6 This study', 43],
 [3, '1.6.1 Background to this study', 43],
 [3, '1.6.2 Aims and theoretical framework', 43],
 [3, '1.6.3 Relation with consultants, other speakers and the community', 45],
 [3, '1.6.4 Data and research methods', 49],
 [3, '1.6.5 Recording and data management', 55],
 [3, '1.6.6 Notation systems', 56],
 [3, '1.6.7 Malay and Indonesian', 58],
 [1, '2 Short grammatical overview of Kalamang', 61],
 [2, '2.1 Phonology', 61],
 [2, '2.2 Morphophonology', 62],
 [2, '2.3 Nouns, noun phr

# Block test

In [57]:
def build_block_table(pdf_file_path):
    block_table = []

    try:
        print("Opening PDF file...")
        doc = pymupdf.open(pdf_file_path)
        print("PDF file opened successfully.")
        print("Number of pages in PDF file:", doc.page_count)
    except Exception as e:
        print("Error occurred while opening PDF file:", e)
        return block_table

    for page_num, page in tqdm(enumerate(doc), total=doc.page_count, desc="Processing pages"):
        blocks = page.get_text("dict")["blocks"]
        for block_idx, block in enumerate(blocks):
            block_text = []
            block_bbox = block.get("bbox", None)
            block_font_sizes = []
            block_fonts = []

            num_spans = 0   # To count number of spans in the block
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span.get("text", "").strip()
                    if text != "":
                        block_text.append(text)
                        block_font_sizes.append(span.get("size", None))
                        block_fonts.append(span.get("font", None))
                        num_spans += 1
            # Get the most common font size and font in the block based on the length of text
            if block_font_sizes and block_text:
                most_common_size = max(
                    collections.Counter(block_font_sizes).items(),
                    key=lambda item: sum(len(block_text[i]) for i, size in enumerate(block_font_sizes) if size == item[0])
                )[0]
            if block_fonts and block_text:
                most_common_font = max(
                    collections.Counter(block_fonts).items(),
                    key=lambda item: sum(len(block_text[i]) for i, font in enumerate(block_fonts) if font == item[0])
                )[0]

            if block_text:
                block_table.append({
                    "page": page_num + 1,
                    "block": block_idx,
                    "text": " ".join(block_text),
                    "bbox": block_bbox,
                    "font_sizes": list(block_font_sizes),
                    "fonts": list(block_fonts),
                    "most_common_size": most_common_size if block_font_sizes else None,
                    "most_common_font": most_common_font if block_fonts else None,
                    "num_lines": len(block.get("lines", [])),
                    "num_spans": num_spans
                })

    return block_table

In [58]:
block_table = build_block_table(pdf_file)
print(f"Extracted {len(block_table)} text blocks from the PDF.")

Opening PDF file...
PDF file opened successfully.
Number of pages in PDF file: 573


Processing pages:  20%|█▉        | 113/573 [00:10<00:41, 11.13it/s]


KeyboardInterrupt: 

In [ ]:
cur_page = -1
for block in block_table:
    if block['page'] != cur_page:
        print(f"Page {block['page']}, Block {block['block']}: {block['text'][:100]}...")
        cur_page = block['page']
    else:
        print(f"        Block {block['block']}: {block['text'][:100]}...")


Page 1, Block 0: A grammar of Kalamang...
        Block 1: Eline Visser...
        Block 2: language science press Comprehensive Grammar Library 4...
Page 2, Block 0: Comprehensive Grammar Library...
        Block 1: Editor: Martin Haspelmath...
        Block 2: In this series:...
        Block 3: 1. Jacques, Guillaume. A grammar of Japhug....
        Block 4: 2. Grimm, Nadine. A grammar of Gyeli....
        Block 5: 3. Maurer-Cecchini, Philippe. A grammar of Tuatschin: A Sursilvan Romansh dialect....
        Block 6: 4. Visser, Eline. A grammar of Kalamang....
        Block 7: This series grew out of the grammars published in Studies in Diversity Linguistics , which are...
        Block 8: proudly mentioned:...
        Block 9: 4. Berghäll, Liisa. A grammar of Mauwake....
        Block 10: 5. Wilbur, Joshua. A grammar of Pite Saami....
        Block 11: 7. Schackow, Diana. A grammar of Yakkha....
        Block 12: 8. Liljegren, Henrik. A grammar of Palula....
        Block 13: 9. Shim

# Dictionary test

In [ ]:
for b in block_table:
    if re.search(r'wordlist', b['text']):
        print(f"Page {b['page']}, Block {b['block']}: {b['text'][:100]}...")


Page 90, Block 11: Figure 3.3: Frequency of vowel phonemes in the Kalamang wordlist (August 2020)...
Page 138, Block 4: di= causative Pred no §11.4.4.1 ko= applicative V no §11.4.3 nak= ‘just’ V no wordlist nau= reciproc...
Page 475, Block 1: Below is a Kalamang-English wordlist with grammatical categories and a rever- sal entries index. A m...
Page 555, Block 0: Schapper, Antoinette & Harald Hammarström. 2013. Innovative numerals in Malayo-Polynesian languages ...
Page 571, Block 7: Wallacea, linguistic, see East Indone- sian languages West Bomberai, 5, 18 wh-question, see question...


In [ ]:
def detect_dictionary_from_spans(block_table, min_lines=5, min_ratio=0.3, debug=False):
    entry_pattern = re.compile(r"^[\wāēīōūḍḥṅñçčśṣžżł]+(?:\s\(.+?\))?\s*[-:–=]\s*[A-Za-z ,;()'‘’\-]+$")
    diacritic_pattern = re.compile(r"[āēīōūḍḥṅñçčśṣžżł]")
    header_pattern = re.compile(r"(?i)\b(dictionary|wordlist|lexicon|vocabulary|glossary|word\s*index)\b")

    # Detect heading blocks
    heading_blocks = [
        b for b in block_table
        if header_pattern.search(b["text"]) and len(b["text"].split()) <= 5
    ]

    candidates = []

    for block in block_table:
        text = block.get("text", "")
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        if not lines:
            continue

        entry_like = sum(1 for l in lines if entry_pattern.match(l))
        diacritic_lines = sum(1 for l in lines if diacritic_pattern.search(l))
        entry_ratio = entry_like / len(lines)
        avg_tokens = sum(len(l.split()) for l in lines) / len(lines)
        diacritic_ratio = diacritic_lines / len(lines)

        score = (
            0.6 * entry_ratio
            + 0.2 * (1 - abs(avg_tokens - 3) / 3)
            + 0.2 * diacritic_ratio
        )

        # Header proximity bonus
        if any(abs(block["page"] - h["page"]) <= 1 for h in heading_blocks):
            score += 0.2

        if len(lines) >= min_lines and entry_ratio >= min_ratio and score > 0.5:
            candidates.append({
                "page": block["page"],
                "block": block["block"],
                "confidence": round(min(score, 1.0), 3),
                "lines": lines,
                "bbox": block.get("bbox"),
                "font": block.get("most_common_font"),
                "size": block.get("most_common_size")
            })

    candidates.sort(key=lambda x: x["confidence"], reverse=True)
    return candidates


In [ ]:
dict_candidates = detect_dictionary_from_spans(block_table, debug=True)

print(f"Detected {len(dict_candidates)} likely dictionary blocks")
for c in dict_candidates[:5]:
    print(f"\nPage {c['page']} Block {c['block']} (confidence={c['confidence']})")
    for line in c['lines'][:10]:
        print("   ", line)

Detected 0 likely dictionary blocks


In [ ]:
import re
import numpy as np



def block_heading_score(block, prev_block=None, next_block=None, 
                        body_size=10.0, median_gap=12.0, page_width=600):
    txt = block["text"].strip()
    fonts = [f.lower() for f in block.get("fonts", [])]
    x, y = block["bbox"][0], block["bbox"][1]

    score = 0.0

    most_common_size = block.get("most_common_size", body_size)
    most_common_font = block.get("most_common_font", "").lower()

    # --- 1. Font & Style (normalized by body size) ---
    size_z = (most_common_size - body_size) / max(1.0, body_size * 0.25)
    if size_z > 0.5:
        score += min(0.4, 0.2 + 0.1 * size_z)  # cap effect

    if most_common_size > body_size:
        score += min(0.4, 0.2 + 0.1 * (most_common_size - body_size) / max(1.0, body_size * 0.25))
    if "bold" in most_common_font:
        score += 0.2
    if "italic" in most_common_font:
        score += 0.1   # smaller weight (glosses often italic)

    # --- 2. Block Length ---
    wc = len(txt.split())
    if 2 <= wc <= 12:
        score += 0.2   # sweet spot for headings
    elif wc > 20:
        score -= 0.2   # too long → body text
    elif wc == 1:
        score += 0.05  # allow "Nouns", "Sandhi"

    # --- 3. Script / Language mix ---
    non_ascii_ratio = sum(ord(c) > 127 for c in txt) / max(1, len(txt))
    if non_ascii_ratio > 0.3 and wc <= 3:
        score -= 0.15  # short foreign snippet → likely example
    elif non_ascii_ratio > 0.3 and wc > 3:
        score += 0.05  # bilingual headings → mild boost

    # --- 4. Vertical spacing (z-score) ---
    if prev_block:
        gap_before = y - prev_block["bbox"][3]
        gap_z = (gap_before - median_gap) / max(1.0, median_gap * 0.25)
        if gap_z > 1.0:
            score += min(0.25, 0.1 + 0.05 * gap_z)

    # --- 5. Alignment ---
    if x < 0.15 * page_width:
        score += 0.1   # flush-left
    elif abs((x + len(txt)*most_common_size*0.4) - page_width/2) < 0.2*page_width:
        score += 0.15  # approximately centered → heading
    else:
        score -= 0.05

    # --- 6. Punctuation / Structure ---
    if txt.endswith("?") and wc > 6:
        score -= 0.2   # long Q → body text
    if txt.startswith(("‘", '"')):
        score -= 0.1   # quoted examples
    # Check for capitalization of first letter
    if txt and txt[0].isupper():
        score += 0.1   # capitalized → likely heading
    else:
        score -= 0.1   # not capitalized → likely body text

    # --- 7. Next block context ---
    if next_block:
        next_txt = next_block["text"].strip()
        next_wc = len(next_txt.split())
        if next_wc >= 8:
            score += 0.2   # paragraph follows
        elif next_wc <= 3 and not next_txt.endswith("."):
            score -= 0.15  # gloss or short label after

    return score

In [ ]:
page_width = doc[0].rect.width
scores = []
blocks_scores = []
for i, block in enumerate(block_table):
    prev_block = block_table[i-1] if i > 0 and block_table[i-1]["page"] == block["page"] else None
    next_block = block_table[i+1] if i < len(block_table)-1 and block_table[i+1]["page"] == block["page"] else None
    score = block_heading_score(block, prev_block, next_block, page_width=page_width)
    scores.append(score)
    blocks_scores.append((block, score))

for block, score in blocks_scores:
    if score < 0.7:
        print(f" - {block['text'][:54]:<54} | Page: {block['page']:>3}, Block: {block['block']:>3}")
        continue
    print("---")
    print(f"{score:.2f} | {block['text'][:50]:<50} | Page: {block['page']:>3}, Block: {block['block']:>3}")


In [ ]:
# Find frequency of different font sizes and calculate the total length of text for each font size
from collections import Counter
font_size_counter = Counter()
font_size_length_counter = Counter()
font_style_size_counter = Counter()
font_style_size_dict = {}


for block, score in blocks_scores:
    for size, font in zip(block.get("font_sizes", []), block.get("fonts", [])):
        if size and font:
            font_size_counter[size] += 1
            font_size_length_counter[size] += len(block.get("text", ""))
            font_style_size_counter[(size, font)] += 1
            font_style_size_dict[(size, font)] = font_style_size_dict.get((size, font), 0) + len(block.get("text", ""))

main_body_font_size, main_body_font_style = font_style_size_counter.most_common()[0][0]

In [ ]:
main_body_font_size, main_body_font_style

In [ ]:
font_size_counter = Counter()
font_size_length_counter = Counter()
font_style_size_counter = Counter()
font_style_size_dict = {}
font_bucket_dict = collections.defaultdict(list)

for block, score in blocks_scores:
    if score < 0.7:
        continue
    font_sizes = block.get("font_sizes", [])
    avg_font_size = sum(font_sizes) / len(font_sizes) if font_sizes else 0
    
    if avg_font_size < main_body_font_size - 0.1:
        continue

    for size, font in zip(block.get("font_sizes", []), block.get("fonts", [])):
        if size and font:
            #font_size_counter[size] += 1
            #font_size_length_counter[size] += len(block.get("text", ""))
            #font_style_size_counter[(size, font)] += 1
            #font_style_size_dict[(size, font)] = font_style_size_dict.get((size, font), 0) + len(block.get("text", ""))
            
            # Add to the bucket dictionary
            font_bucket_dict[(size, font, block["bbox"][0])].append((block["text"], score))


# Round x and size values within their respective thresholds into the same bucket
rounded_font_bucket_dict = collections.defaultdict(list)

round_x_threshold = 2.0
round_size_threshold = 0.5

for (size, font, x), texts_scores in font_bucket_dict.items():
    rounded_x = round(x / round_x_threshold) * round_x_threshold  # Round x to the nearest round_x_threshold
    rounded_size = round(size / round_size_threshold) * round_size_threshold  # Round size to the nearest round_size_threshold
    rounded_font_bucket_dict[(rounded_size, font, rounded_x)].extend(texts_scores)

font_bucket_dict = rounded_font_bucket_dict

font_bucket_dict_2 = {}

# Display the font buckets with average scores
for (size, font, x), texts_scores in font_bucket_dict.items():
    combined_text = "\n - ".join(text for text, _ in texts_scores)
    avg_score = sum(score for _, score in texts_scores) / len(texts_scores)
    print(f"Font Size: {size}, Font: {font}, X: {x}")
    print(f"Combined Text: {combined_text[:100]}")
    print(f"Average Score: {avg_score:.2f}")
    print("-" * 80)
    font_bucket_dict_2[(size,font,x)] = (combined_text, avg_score)


In [ ]:
# Display the font bucket with most to least avg score
for (size, font, x), (combined_text, avg_score) in sorted(font_bucket_dict_2.items(), key=lambda item: item[1][1], reverse=True):
    print(f"Font Size: {size}, Font: {font}, X: {x}")
    print(f"Combined Text: \n- {combined_text}")
    print(f"Average Score: {avg_score:.2f}")
    print("-" * 80)

In [ ]:
import ollama



In [ ]:
from collections import defaultdict

# Create a hierarchical bucket structure
hierarchical_buckets = defaultdict(lambda: defaultdict(list))

# Iterate through font_bucket_dict_2
for (font_size, font_style, x), (combined_text, avg_score) in font_bucket_dict_2.items():
    hierarchical_buckets[(font_size, font_style)][x].append({
        "combined_text": combined_text,
        "avg_score": avg_score
    })

# Display the hierarchical buckets
for (font_size, font_style), x_buckets in hierarchical_buckets.items():
    print(f"Font Size: {font_size}, Font Style: {font_style}")
    for x, entries in x_buckets.items():
        print(f"  X: {x}")
        for entry in entries:
            print(f"    Combined Text: {entry['combined_text'][:100]}")  # Truncate text for readability
            print(f"    Average Score: {entry['avg_score']:.2f}")
        print("-" * 80)
    print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt

# Extract font sizes and their corresponding total lengths
font_sizes = [key[0] for key in font_style_size_dict.keys()]
total_lengths = list(font_style_size_dict.values())

# Create the histogram
plt.figure(figsize=(10, 6))
plt.bar(font_sizes, total_lengths, color='blue', alpha=0.7)
plt.title('Histogram of Font Sizes and Total Text Lengths')
plt.xlabel('Font Size')
plt.ylabel('Total Text Length')
plt.grid(axis='y', alpha=0.75)
plt.xticks(rotation=45)
plt.show()

In [ ]:
main_body_font_size, main_body_font_style = font_style_size_counter.most_common()[0]

In [ ]:
doc = pymupdf.open(pdf_file)

for block_dict, score in tqdm(blocks_scores):
    if score < 0.7:
        continue
    page_num = block_dict["page"] - 1  # PyMuPDF is 0-based
    page = doc[page_num]
    rect = pymupdf.Rect(block_dict["bbox"])

    #if font size is less than main body font size, skip
    font_sizes = block_dict.get("font_sizes", [])
    if not font_sizes:
        continue

    if np.mean(font_sizes) < main_body_font_size[0]:
        continue    

    # Draw rectangle around the block
    page.draw_rect(rect, color=(1, 0, 0), width=1)

    # Optionally, overlay score value
    page.insert_text((rect.x0, rect.y0 - 2),
                     f"{score:.2f}",
                     fontsize=6,
                     color=(0, 0, 1))
    

doc.save("scored_blocks.pdf")


In [ ]:
base_prompt_heading = """
You are a strict classifier for PDF headings from grammar books.

Your task: For the collection of candidate headings of the same font style, size and x position below, decide if the collection of texts is predominantly HEADING or NOT_HEADING.
You must classify based on the following rules and examples. These headings were extracted from a scoring function that identified likely headings based on font size, style, position, and other heuristics.

Rules for HEADING:
- Short title (< 10 words).
- May have numbering ("1.", "2.3", "CHAPTER 4").
- Often Title Case or ALL CAPS.
- Does NOT end with a period, semicolon, gloss markers, or quotes.
- Introduces a new section.

Rules for NOT_HEADING:
- Running prose sentences.
- Linguistic examples or glosses (hyphens like "eat-1SG", equals signs, gloss abbreviations SG/PL/NOM/PST/etc.).
- Single words in foreign script or transliteration (like "māṣma").
- Translations in quotes ("I eat.").

Examples:
"Verb Morphology" → HEADING
"CHAPTER 3: Syntax" → HEADING
"In this section we discuss case alignment." → NOT_HEADING
"1SG eat-PST" → NOT_HEADING
"māṣma" → NOT_HEADING
"‘I eat.’" → NOT_HEADING

Now classify the below collection of candidate headings. Provide ONLY "HEADING" or "NOT_HEADING" as your answer for the entire list.
Text List:
{text_list}
"""

In [ ]:
import ollama

model = "qwen3:1.7b"

def classify_lines(lines, model="qwen3:1.7b"):
    text_list = "\n - ".join(lines)
    prompt = base_prompt_heading.format(text_list=text_list)
    
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    # Remove <think> tag and text in between
    #print(response)
    message = response.get("message", {})
    if not message:
        return "NO ANSWER"
    answer = message.get("content", "").strip()
    answer = answer[answer.find("</think>") + len("</think>"):].strip()
    
    print(answer)
    if "HEADING" in answer.upper() and "NOT_HEADING" not in answer.upper():
        return "HEADING"
    elif "NOT_HEADING" in answer.upper():
        return "NOT_HEADING"
    else:
        return "UNSURE: " + answer

for (size, font, x), (combined_text, avg_score) in sorted(font_bucket_dict_2.items(), key=lambda item: item[1][1], reverse=True):
    lines = combined_text.split("\n - ")
    classification = classify_lines(lines, model=model)
    print(f"Font Size: {size}, Font: {font}, X: {x}")
    print(f"Classification: {classification}")
    print(f"Combined Text: \n- {combined_text}")
    print(f"Average Score: {avg_score:.2f}")
    print("-" * 80)
    
        
    

# Actual code

In [ ]:
def build_line_table(pdf_file_path, y_tolerance=2.0, line_tolerance=2.0):
    all_lines = []

    try:
        print("Opening PDF file...")
        doc = pymupdf.open(pdf_file_path)
        print("PDF file opened successfully.")
        print("Number of pages in PDF file:", doc.page_count)
    except Exception as e:
        print("Error occurred while opening PDF file:", e)
        return all_lines

    for page_num, page in tqdm(
        enumerate(doc), total=doc.page_count, desc="Processing pages"
    ):
        blocks = page.get_text("dict")["blocks"]
        for block_idx, block in enumerate(blocks):
            for line_idx, line in enumerate(block.get("lines", [])):
                spans = line.get("spans", [])
                if not spans:
                    continue

                # Sort spans by y (top to bottom) and then x (left to right) with tolerance
                # This is to handle cases where spans are not in reading order
                for span in spans:
                    x0, y0, _, _ = span["bbox"]
                    span["_y_round"] = round(y0 / y_tolerance) * y_tolerance
                    span["_x"] = x0
                spans_sorted = sorted(spans, key=lambda s: (s["_y_round"], s["_x"]))
                # spans = spans_sorted
                # Merge spans into one line string
                merged_text = " ".join(
                    span["text"].strip()
                    for span in spans_sorted
                    if span["text"].strip()
                )
                if not merged_text:
                    continue

                # Choose a "dominant" font/size for the line
                sizes = [span["size"] for span in spans_sorted]
                fonts = [span["font"] for span in spans_sorted]

                # Representative = most common size & font
                rep_size = max(set(sizes), key=sizes.count)
                rep_font = max(set(fonts), key=fonts.count)

                # Position = from first span
                x, y = spans_sorted[0]["bbox"][:2]

                all_lines.append(
                    {
                        "page": page_num + 1,
                        "block": block_idx,
                        "line": line_idx,
                        "text": merged_text,
                        "size": rep_size,
                        "font": rep_font,
                        "bbox": spans_sorted[0]["bbox"],
                        "x": x,
                        "y": y,
                    }
                )

    # Sort all lines by page, y (top to bottom), x (left to right)
    #

    for line in all_lines:
        line["_y_round"] = round(line["y"] / line_tolerance) * line_tolerance
    all_lines.sort(key=lambda l: (l["page"], l["_y_round"], l["x"]))

    return all_lines



In [ ]:
all_lines = build_line_table(pdf_file)

In [ ]:
all_lines

# New heading test

In [ ]:
import re
import numpy as np

def heading_score(line, prev_line=None, next_line=None, 
                  body_size=10.0, median_gap=12.0, page_width=600):
    txt   = line["text"].strip()
    sz    = line["size"]
    font  = line["font"].lower()
    x     = line["x"]
    y     = line["y"]

    score = 0.0

    # --- 1. Font & Style (normalized by body size) ---
    size_z = (sz - body_size) / max(1.0, body_size * 0.25)
    if size_z > 0.5:
        score += min(0.4, 0.2 + 0.1 * size_z)  # cap effect
    if "bold" in font:
        score += 0.2
    if "italic" in font:
        score += 0.1   # smaller weight (glosses often italic)

    # --- 2. Line Length ---
    wc = len(txt.split())
    if 2 <= wc <= 12:
        score += 0.2   # sweet spot for headings
    elif wc > 20:
        score -= 0.2   # too long → body text
    elif wc == 1:
        score += 0.05  # allow "Nouns", "Sandhi"

    # --- 3. Script / Language mix ---
    non_ascii_ratio = sum(ord(c) > 127 for c in txt) / max(1, len(txt))
    if non_ascii_ratio > 0.3 and wc <= 3:
        score -= 0.15  # short foreign snippet → likely example
    elif non_ascii_ratio > 0.3 and wc > 3:
        score += 0.05  # bilingual headings → mild boost

    # --- 4. Vertical spacing (z-score) ---
    if prev_line:
        gap_before = y - prev_line["y"]
        gap_z = (gap_before - median_gap) / max(1.0, median_gap * 0.25)
        if gap_z > 1.0:
            score += min(0.25, 0.1 + 0.05 * gap_z)

    # --- 5. Alignment ---
    if x < 0.15 * page_width:
        score += 0.1   # flush-left
    elif abs((x + len(txt)*sz*0.4) - page_width/2) < 0.2*page_width:
        score += 0.15  # approximately centered → heading
    else:
        score -= 0.05

    # --- 6. Punctuation / Structure ---
    if txt.endswith("?") and wc > 6:
        score -= 0.2   # long Q → body text
    if txt.startswith(("‘", '"')):
        score -= 0.1   # quoted examples
    # Check for capitalization of first letter
    if txt and txt[0].isupper():
        score += 0.1   # capitalized → likely heading
    else:
        score -= 0.1   # not capitalized → likely body text

    # --- 7. Next line context ---
    if next_line:
        next_txt = next_line["text"].strip()
        next_wc = len(next_txt.split())
        if next_wc >= 8:
            score += 0.2   # paragraph follows
        elif next_wc <= 3 and not next_txt.endswith("."):
            score -= 0.15  # gloss or short label after

    return score


In [ ]:
#

In [ ]:
scores = []
lines_scores = []
for i, line in enumerate(all_lines):
    prev_line = all_lines[i-1] if i > 0 else None
    next_line = all_lines[i+1] if i < len(all_lines)-1 else None
    s = heading_score(line, prev_line, next_line, body_size=9.96, median_gap=13)
    scores.append((line["text"], round(s, 2)))
    lines_scores.append((line, round(s, 2)))

# Find mean, stddev scores
mean_score = np.mean([s for _, s in scores])
stddev_score = np.std([s for _, s in scores])

print(f"Mean heading score: {mean_score:.2f}")
print(f"Standard deviation of heading scores: {stddev_score:.2f}")

for text, score in scores:
    if score >= 0.8:
        print(f"{score:.2f} | {text}")


In [ ]:
for line, score in lines_scores:
    if score < 0.7:
        print(f" - {line['text'][:54]:<54} | Page: {line['page']:>3}, Block: {line['block']:>3}, Line: {line['line']:>3}")
        continue
    print("---")
    print(f"{score:.2f} | {line['text'][:50]:<50} | Page: {line['page']:>3}, Block: {line['block']:>3}, Line: {line['line']:>3}")

In [ ]:
all_lines[0]

In [ ]:

doc = pymupdf.open(pdf_file)

for line_dict, score in tqdm(lines_scores):
    if score < 0.7:
        continue
    page_num = line_dict["page"] - 1  # PyMuPDF is 0-based
    page = doc[page_num]


    bbox = line_dict["bbox"]
    rect = pymupdf.Rect(bbox)

    # Normalize score into [0,1] for coloring
    norm_score = min(max(score, 0), 1)

    # Color gradient: green (low) → red (high)
    color = (norm_score, 1 - norm_score, 0)

    # Draw transparent rectangle over the line
    page.draw_rect(rect, color=color, fill=color, fill_opacity=0.25)

    # Optionally, overlay score value
    page.insert_text((rect.x0, rect.y0 - 2),
                     f"{score:.2f}",
                     fontsize=6,
                     color=(0, 0, 1))

    # IF the line id is 0, indicating start of a block, draw a thicker line above
    if line_dict["line"] == 0:
        page.draw_line((rect.x0, rect.y0 - 1), (rect.x1, rect.y0 - 1), color=(0, 0, 0), width=1.5)


# Save annotated PDF
doc.save("scored_lines.pdf")

In [ ]:
from collections import Counter

# Initialize a counter for headings
heading_counter = Counter()
heading_scores = Counter()

# Iterate through lines and scores
for line, score in lines_scores:
    if score > 0.6:  # Check if the score is above 0.6
        fsize = line["size"]
        fstyle = line["font"].lower()
        heading_counter[(fsize, fstyle)] += 1
        heading_scores[(fsize, fstyle)] += score

heading_scores_average = {
    k: v / heading_counter[k] for k, v in heading_scores.items()
}

# Sort heading_scores_average
heading_scores_average = dict(sorted(heading_scores_average.items(), key=lambda item: item[1], reverse=True))

# Print the counter
for (size, style), count in heading_counter.most_common()[:10]:
    print(f"Size: {size}, Style: {style}, Count: {count}")

print("---")
# Print the scores
for (size, style), score in heading_scores_average.items():
    print(f"Size: {size}, Style: {style}, Average Score: {score:.2f}")

# Failed heading test

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def extract_line_features(line):
    """Return basic features for clustering + scoring"""
    txt = line["text"].strip()
    font = line["font"].lower()
    return {
        "font_size": line["size"],
        "is_bold": int("bold" in font),
        "is_italic": int("italic" in font),
        "text": txt,
        "x": line["x"],
        "y": line["y"],
        "num_words": len(txt.split()),
        "is_all_caps": int(txt.isupper()),
        "ends_with_punct": int(txt.endswith((".", ";", ":"))),
    }


In [ ]:
def cluster_fonts(all_lines, n_clusters=3):
    """Cluster lines by font size + style"""
    feats = []
    for l in all_lines:
        f = extract_line_features(l)
        feats.append([f["font_size"], f["is_bold"], f["is_italic"]])
    feats = np.array(feats)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(feats)

    # Attach cluster labels back to lines
    for l, lbl in zip(all_lines, labels):
        l["cluster"] = int(lbl)

    # Determine "main body cluster" = cluster with most lines
    counts = np.bincount(labels)
    body_cluster = int(np.argmax(counts))

    return body_cluster


In [ ]:
def heading_score(line, body_cluster, main_body_size, prev_line=None, next_line=None):
    f = extract_line_features(line)
    score = 0

    

    # Size
    if f["font_size"] > main_body_size + 0.5:
        score += (f["font_size"] - main_body_size) / 2.0
    if f["font_size"] < main_body_size - 0.5:
        score -= 10  # unlikely to be heading
        return score
    
    # If line is NOT in body cluster → strong candidate
    if line["cluster"] != body_cluster:
        score += 2

    # Typography
    if f["is_bold"]:
        score += 1
    if f["is_all_caps"]:
        score += 1

    # Textual
    if f["num_words"] <= 6:
        score += 1
    if not f["ends_with_punct"]:
        score += 0.5
    if f["text"][:3].isdigit() or f["text"].startswith(("§", "Chapter", "CHAPTER")):
        score += 1.5

    # Layout (optional: needs prev/next line info)
    if prev_line:
        gap = line["y"] - prev_line["y"]
        if gap > 1.5 * (prev_line.get("line_gap", 12)):
            score += 1

    if next_line:
        gap_next = next_line["y"] - line["y"]
        if gap_next > 1.5 * (line.get("line_gap", 12)):
            score += 0.5

    return score


In [ ]:
def detect_heading_candidates(all_lines, main_body_size, score_threshold=3.0):
    body_cluster = cluster_fonts(all_lines)

    candidates = []
    for i, line in enumerate(all_lines):
        prev_line = all_lines[i-1] if i > 0 else None
        next_line = all_lines[i+1] if i < len(all_lines)-1 else None
        score = heading_score(line, body_cluster, main_body_size,prev_line, next_line)

        line["heading_score"] = score
        line["is_heading_candidate"] = score >= score_threshold

        if line["is_heading_candidate"]:
            candidates.append(line)

    return candidates


In [ ]:

candidates = detect_heading_candidates(all_lines, main_body_size)

for c in candidates:
    if c["heading_score"] >= 3.0:
        print(f"Page {c['page']:03d}: {c['text']} (score={c['heading_score']:.1f}, cluster={c['cluster']})")


In [ ]:
base_prompt_heading = """
You are a strict classifier for PDF lines from grammar books.

Your task: For each line below, decide if it is a SECTION HEADING or NOT.

Rules for HEADING:
- Short title (< 10 words).
- May have numbering ("1.", "2.3", "CHAPTER 4").
- Often Title Case or ALL CAPS.
- Does NOT end with a period, semicolon, gloss markers, or quotes.
- Introduces a new section.

Rules for NOT_HEADING:
- Running prose sentences.
- Linguistic examples or glosses (hyphens like "eat-1SG", equals signs, gloss abbreviations SG/PL/NOM/PST/etc.).
- Single words in foreign script or transliteration (like "māṣma").
- Translations in quotes ("I eat.").

Examples:
"Verb Morphology" → HEADING
"CHAPTER 3: Syntax" → HEADING
"In this section we discuss case alignment." → NOT_HEADING
"1SG eat-PST" → NOT_HEADING
"māṣma" → NOT_HEADING
"‘I eat.’" → NOT_HEADING

Now classify these lines. Return JSON only, like:
{"0": "HEADING", "1": "NOT_HEADING", ...}

Lines:
0: Verb Morphology
1: 1SG eat-PST
2: In this section we discuss case alignment
3: CHAPTER 3: Syntax

"""

In [ ]:
import ollama

model = "qwen3:1.7b"

def classify_line(line):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": base_prompt_heading},
            {"role": "user", "content": f'Line: "{line}"'}
        ]
    )
    return response["message"]["content"].strip()

# Test batch
lines = [
    "Verb Morphology",
    "1SG eat-PST",
    "In this section we discuss case alignment",
    "CHAPTER 3: Syntax"
]


for l in lines:
    print(l, "→", classify_line(l))

In [ ]:
import requests
import json
import csv
from itertools import islice

MODEL = "qwen3:1.7b"
OLLAMA_URL = "http://localhost:11434/api/generate"

def chunks(iterable, n=20):
    """Yield successive n-sized chunks from iterable."""
    it = iter(iterable)
    for first in it:
        yield [first] + list(islice(it, n - 1))

def classify_batch(lines, model=MODEL):
    # Numbered list for the prompt
    numbered = "\n".join(f"{i}: {l['text']}" for i, l in enumerate(lines))
    prompt = f"""
You are a strict classifier for PDF lines from grammar books.

Task: For each line, decide if it is a SECTION HEADING or NOT.

HEADING characteristics:
- Short title (< 10 words).
- May be numbered ("1.", "2.3", "CHAPTER 4").
- Often Title Case or ALL CAPS.
- No sentence-ending punctuation.
- Stands alone and introduces a section.

NOT_HEADING characteristics:
- Running prose sentences.
- Linguistic examples or glosses (hyphens like "eat-1SG", equals signs, gloss abbreviations SG/PL/NOM/PST/etc).
- Single words in transliteration or non-English script.
- Translations in quotes ("I eat.").

Return JSON ONLY in this form:
{{"0": "HEADING", "1": "NOT_HEADING", ...}}

Lines:
{numbered}
"""
    #print(numbered)
    r = requests.post(
        OLLAMA_URL,
        json={"model": model, "prompt": prompt, "stream": False},
    )
    resp = r.json()["response"].strip()
    
    #print(resp)
    # Clean response. Remove everything before </think> and strip
    resp = resp[resp.find("</think>") + len("</think>"):].strip()
    resp = resp[resp.find("{"): resp.rfind("}") + 1]
    # Try to parse JSON, fallback to eval if formatting glitches
    try:
        result = json.loads(resp)
    except Exception:
        resp = resp[resp.find("{"): resp.rfind("}") + 1]
        result = json.loads(resp)

    return result

def classify_all_lines(all_lines, output_csv="classified_lines.csv", batch_size=20):
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["page", "text", "heading_label"])

        for batch in tqdm(chunks(all_lines, n=batch_size)):
            result = classify_batch(batch)
            for i, line in enumerate(batch):
                label = result.get(str(i), "NOT_HEADING")
                writer.writerow([line["page"], line["text"], label])

    print(f"✅ Saved classified lines to {output_csv}")


In [ ]:
# After building your line table
# all_lines = build_line_table("my_grammar.pdf")
classify_all_lines(all_lines, output_csv="headings.csv", batch_size=20)


# Analyse fonts


In [ ]:


def analyze_fonts(all_lines, min_heading_occ_count=5, min_heading_total_char_length=30,main_body_tolerance = 0.5):

    size_occurrences = collections.Counter(line["size"] for line in all_lines)
    # Get lengths of text above main body size for headings total
    length_by_size = collections.Counter()
    for line in all_lines:
        length_by_size[line["size"]] += len(line["text"])

    main_body_size = max(length_by_size, key=length_by_size.get)
    print("Determined main body font size:", main_body_size)

    headings = []
    for size,total_length in length_by_size.most_common():
        if size>main_body_size+main_body_tolerance and total_length >= min_heading_total_char_length and size_occurrences[size] >= min_heading_occ_count:
            print(f"  Possible heading size: {size} (total text length: {total_length} chars, occurrences: {size_occurrences[size]})")
            headings.append((size, total_length, size_occurrences[size]))
    headings_sorted = [size for size,_,_ in sorted(headings, reverse=True)]



    return main_body_size, headings_sorted

main_body_size, headings_sorted = analyze_fonts(all_lines)
main_body_size, headings_sorted

In [ ]:
def build_sections(all_lines, main_body_size, headings_sizes):
    sections = []
    current = None

    for line in all_lines:
        sz, txt, font = line["size"], line["text"], line["font"]
        if sz in headings_sizes:
            if current:
                sections.append(current)
            current = {
                "heading": txt,
                "heading_level": headings_sizes.index(sz) + 1,
                "text": "",
                "font_size": sz,
                "font_style": font,
                "page": line["page"],
            }
        elif current and abs(sz - main_body_size) <= 0.5:
            if current["text"]:
                current["text"] += " "
            current["text"] += txt

    if current:
        sections.append(current)

    return sections

sections = build_sections(all_lines, main_body_size, headings_sorted)

In [ ]:
sections

In [ ]:
def clean_sections(sections):
    
    for i, section in enumerate(sections):
        if section is None:
            continue
        
        # merge consecutive empty same-level headings
        if i < len(sections) - 1 and section["heading_level"] == sections[i + 1]["heading_level"] and section["text"].strip() == "":
            print("Merging empty headings:", section["heading"], " - ")
            while (i < len(sections) - 1 and
                sections[i + 1] and
                section["heading_level"] == sections[i + 1]["heading_level"] and
                sections[i + 1]["text"].strip() == ""):
            
                section["heading"] += " " + sections[i + 1]["heading"]
                print(" - ", sections[i + 1]["heading"])
                sections[i + 1] = None
                i += 1

            # only copy text if the *next* section actually has body text
            if i < len(sections) - 1 and sections[i + 1] and sections[i + 1]["text"].strip() != "":
                section["text"] = sections[i + 1]["text"]
                section["heading"] += " " + sections[i + 1]["heading"]
                print(" -> ", sections[i + 1]["heading"])
                sections[i + 1] = None

    sections_filtered = [s for s in sections if s is not None]
    return sections_filtered

sections_cleaned = clean_sections(sections)
#sections
#sections_cleaned

        

In [ ]:
sections_cleaned

In [ ]:
from langdetect import detect_langs
import re

_SPLIT_RE = re.compile(r"[\s.\-=:\/]+")

def classify_text_igt_english(text: str) -> str:
    if not text or not text.strip():
        return "reject"

    IGT_ABBRS = {
        "sg","pl","du","pst","prs","fut","ipfv","pfv","prog","hab",
        "nom","acc","erg","gen","dat","loc","all","abl","ins","com","ben","top","foc",
        "caus","pass","mid","refl","recip","aux","cop","neg","cond","imp","subj","ind"
    }

    tokens = [t for t in _SPLIT_RE.split(text) if t]
    total = len(tokens)
    if total == 0:
        return "reject"

    gloss_hits = sum(2 for t in tokens if t.lower() in IGT_ABBRS or re.fullmatch(r"[123](sg|pl|du)", t.lower()))
    normalized_gloss = gloss_hits / total

    cleaned = re.sub(r"\[.*?\]", "", text)
    english_score = 0.0
    try:
        probs = detect_langs(cleaned)
        if probs and probs[0].lang == "en":
            english_score = probs[0].prob
    except Exception:
        pass

    if normalized_gloss > 0.10:
        return "IGT"
    if len(cleaned) >= 20 and english_score >= 0.70:
        return "English"
    if len(cleaned) >= 40 and english_score >= 0.55:
        return "English"
    return "unknown"

In [ ]:
MARK_RE = re.compile(r"\(\d+\)")

def extract_parallel_sentences(all_lines):
    parallel_sents = []
    idx, N = 0, len(all_lines)
    while idx < N:
        line = all_lines[idx]
        if MARK_RE.fullmatch(line["text"]):
            col_x = None
            rows, j = [], idx + 1
            while j < N:
                ln = all_lines[j]
                if MARK_RE.fullmatch(ln["text"]):
                    break
                if col_x is None:
                    col_x = ln["x"]
                if (col_x - ln["x"]) > 3:
                    break
                rows.append((ln["y"], ln["text"]))
                j += 1

            rows.sort(key=lambda t: t[0])
            grouped, cur_y, buf = [], None, []
            for y, t in rows:
                if cur_y is None or abs(y - cur_y) <= 2:
                    cur_y = y if cur_y is None else (cur_y + y) / 2
                    buf.append(t)
                else:
                    grouped.append(" ".join(buf))
                    cur_y, buf = y, [t]
            if buf:
                grouped.append(" ".join(buf))

            igt, eng, other = [], [], []
            for k, g in enumerate(grouped):
                res = classify_text_igt_english(g)
                if res == "IGT":
                    igt.append(g)
                elif res == "English":
                    eng.extend(grouped[k:])
                    break
                elif res != "reject":
                    other.append(g)

            if eng and other:
                parallel_sents.append({
                    "page": line["page"],
                    "text": " ".join(other),
                    "IGT": " ".join(igt),
                    "English": " ".join(eng),
                })
            idx = j
        else:
            idx += 1
    return parallel_sents

In [ ]:
parallel_sents = extract_parallel_sentences(all_lines)


In [ ]:
parallel_sents

In [ ]:
import spacy

def enrich_parallel_sentences(parallel_sents):
    try:
        nlp = spacy.load("en_core_web_sm")
    except Exception:
        return parallel_sents  # skip if model missing

    for sent in parallel_sents:
        doc = nlp(sent["English"])
        res, tense, number, genders = [], None, None, []
        plural_details, genders_details = None, []

        for tok in doc:
            morph = tok.morph.to_dict()
            res.append({
                "word": tok.text,
                "lemma": tok.lemma_,
                "upos": tok.pos_,
                "tag": tok.tag_,
                "morph": morph,
            })
            if not number and morph.get("Number"):
                number = morph["Number"]
            elif morph.get("Number") == "Plur":
                number = "Plur"
                plural_details = {"word": tok.text, "morph": morph}
            if not tense and morph.get("Tense"):
                tense = morph["Tense"]
            if morph.get("Gender"):
                genders.append(morph["Gender"])
                genders_details.append({"word": tok.text, "morph": morph})

        sent["spacy_info"] = {
            "res": res,
            "tense": tense,
            "number": number,
            "plural_details": plural_details,
            "genders": genders,
            "genders_details": genders_details,
        }
        sent["sentence"] = sent.pop("text")
        sent["translated_sentence"] = sent.pop("English")

    return parallel_sents



In [ ]:
parallel_sents_enhanced = enrich_parallel_sentences(parallel_sents)
parallel_sents_enhanced

# Old code


In [82]:
# Read doc
import pymupdf

pdf_file = "Grammar_books/kalamang.pdf"

doc = pymupdf.open(pdf_file)

In [83]:
section_text = []  # List of sections
current_section = None
text_sizes = []
size_font_tuples = []
text_sizes_combined = []

dictionary = {}
dictionary_with_metadata = {}

for page_num, page in enumerate(doc):
    if page_num< 475:
        continue
    blocks = page.get_text("dict")["blocks"]
    #print(f"--- Page {page_num+1} ---")
    for block in blocks:
        lines = block.get("lines", [])
        for line_index,line in enumerate(lines):
            spans = line.get("spans", [])

            span_index = 0
            while span_index < len(spans):
                span = spans[span_index]
                text = span["text"].strip()
                size = span["size"]
                font = span["font"]
                if not text:
                    span_index += 1
                    continue

                #print(text, size, font)

                if "Italic" in font:
                    #print(text, size, font)
                    translation = spans[span_index + 1]["text"].strip() if span_index + 1 < len(spans) else ""
                    if translation=="":
                        span_index += 1
                        continue

                    # quick check only for kalamang, skip the first POS Tag info
                    if page_num <=514:
                        translation = " ".join(translation.split(" ")[1:])
            
                        if dictionary.get(text) and dictionary[text]!=translation:
                            print(f"Conflict for {text}: '{dictionary[text]}' vs '{translation}'")
                            dictionary[text] += " ; " + translation
                            dictionary_with_metadata[text]["senses"].append({"translation": translation, "page_num": page_num+1})
                        else:
                            translations = translation.split(";")
                            for tr in translations:
                                tr = tr.strip()
                                if tr:
                                    if dictionary.get(text) and dictionary[text] != tr:
                                        print(f"Conflict for '{text}': '{dictionary[text]}' vs '{tr}'")
                                        dictionary[text] += " ; " + tr
                                        dictionary_with_metadata[text]["senses"].append({"translation": tr, "page_num": page_num+1})
                                    else:
                                        dictionary[text] = tr
                                        dictionary_with_metadata[text] = {
                                            "senses":[
                                                {"translation": tr
                                                , "page_num": page_num+1}
                                                    ]
                                            }
                    else:
                        translations = translation.split(";")
                        for tr in translations:
                            tr = tr.strip()
                            if tr:
                                if dictionary.get(tr) and dictionary[tr] != text:
                                    print(f"Conflict for '{tr}': '{dictionary[tr]}' vs '{text}'")
                                    dictionary[tr] += " ; " + text
                                    dictionary_with_metadata[tr]["senses"].append({"translation": text, "page_num": page_num+1})
                                else:
                                    texts = text.split(";")
                                    for text in texts:
                                        text = text.strip()
                                        if text:
                                            if dictionary.get(tr) and dictionary[tr] != text:
                                                print(f"Conflict for '{tr}': '{dictionary[tr]}' vs '{text}'")
                                                dictionary[tr] += " ; " + text
                                                dictionary_with_metadata[tr]["senses"].append({"translation": text, "page_num": page_num+1})
                                            else:
                                                dictionary[tr] = text
                                                dictionary_with_metadata[tr] = {
                                                    "senses":[
                                                        {"translation": text
                                                        , "page_num": page_num+1}
                                                    ]
                                                }
                    span_index += 1


                #print(text, size, font)    
                span_index += 1

                

                #text_sizes.append(size)
                #size_font_tuples.append((size, font))

                
                    

                #print(text,size,font)
                

len(dictionary)

                
               
                    


Conflict for a: 'interjection' vs 'filler'
Conflict for *al: 'classifier for strips' vs 'string'
Conflict for 'anti': 'antidote' vs 'resistant'
Conflict for ar: 'expel' vs 'dive'
Conflict for ar: 'expel ; dive' vs 'sound'
Conflict for ar: 'expel ; dive ; sound' vs 'make a sound'
Conflict for *ar: 'stem' vs 'classifier for stems'
Conflict for 'arep': 'pond' vs 'bay'
Conflict for '=barak': 'too' vs 'any'
Conflict for '=barak': 'too ; any' vs 'even'
Conflict for 'bol': 'mouth' vs 'rim'
Conflict for 'bonasau': 'do' vs 'try'
Conflict for 'buok': 'betel' vs 'betel nut'
Conflict for -ca: 'your (sg)' vs 'man'
Conflict for cam: 'take care of' vs 'tree'
Conflict for 'canam': 'man' vs 'male'
Conflict for cerita: 'tell' vs 'story'
Conflict for cicaun: 'small one' vs 'small'
Conflict for 'deir': 'push' vs 'bring'
Conflict for desil: 'planing tool' vs 'plane'
Conflict for 'dodon': 'things' vs 'clothes'
Conflict for doka: 'sit and do nothing' vs 'heron'
Conflict for 'don iriskap': 'sugar' vs 'white c

2774

In [84]:
dictionary_with_metadata

{'=a': {'senses': [{'translation': 'focus marker', 'page_num': 476}]},
 'a': {'senses': [{'translation': 'interjection', 'page_num': 476},
   {'translation': 'filler', 'page_num': 476}]},
 'a’a': {'senses': [{'translation': 'yes', 'page_num': 540}]},
 'adat': {'senses': [{'translation': 'tradition', 'page_num': 538}]},
 'ade': {'senses': [{'translation': 'pejorative interjection',
    'page_num': 476}]},
 'adi': {'senses': [{'translation': 'interjection of pain', 'page_num': 476}]},
 'adu': {'senses': [{'translation': 'interjection of surprise or pain',
    'page_num': 476}]},
 'afukat': {'senses': [{'translation': 'avocado', 'page_num': 476}]},
 'ahat': {'senses': [{'translation': 'sunday', 'page_num': 536}]},
 '-ahutak': {'senses': [{'translation': 'alone', 'page_num': 476}]},
 'ajar': {'senses': [{'translation': 'v teach () v continue', 'page_num': 476},
   {'translation': 'continue', 'page_num': 519},
   {'translation': 'teach', 'page_num': 537}]},
 'ak': {'senses': [{'translation'

# Dictionary metadata enhancement

In [85]:
for key,value in dictionary_with_metadata.items():
    print(f"{key} -> {value}")

=a -> {'senses': [{'translation': 'focus marker', 'page_num': 476}]}
a -> {'senses': [{'translation': 'interjection', 'page_num': 476}, {'translation': 'filler', 'page_num': 476}]}
a’a -> {'senses': [{'translation': 'yes', 'page_num': 540}]}
adat -> {'senses': [{'translation': 'tradition', 'page_num': 538}]}
ade -> {'senses': [{'translation': 'pejorative interjection', 'page_num': 476}]}
adi -> {'senses': [{'translation': 'interjection of pain', 'page_num': 476}]}
adu -> {'senses': [{'translation': 'interjection of surprise or pain', 'page_num': 476}]}
afukat -> {'senses': [{'translation': 'avocado', 'page_num': 476}]}
ahat -> {'senses': [{'translation': 'sunday', 'page_num': 536}]}
-ahutak -> {'senses': [{'translation': 'alone', 'page_num': 476}]}
ajar -> {'senses': [{'translation': 'v teach () v continue', 'page_num': 476}, {'translation': 'continue', 'page_num': 519}, {'translation': 'teach', 'page_num': 537}]}
ak -> {'senses': [{'translation': 'sea-side', 'page_num': 533}]}
akal ->

In [86]:
# For each entry's sense, try to get spacy analysis if English
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    print("Download and install the 'en_core_web_sm' model for spaCy to enable English analysis.")
    

In [87]:
import re

def clean_gloss_text(gloss: str) -> str:
    # Normalize whitespace
    gloss = re.sub(r"\s+", " ", gloss).strip()

    # Expand common abbreviations
    gloss = re.sub(r"\bk\.?\s*o\.?\b", "kind of", gloss, flags=re.IGNORECASE)
    gloss = re.sub(r"\bsp\.?\b", "species", gloss, flags=re.IGNORECASE)

    # Remove parenthetical grammatical markers, keep content if lexical
    gloss = re.sub(r"\((1|2|3)?(sg|pl)?( poss| pron| refl)?\)", "", gloss, flags=re.IGNORECASE)
    gloss = re.sub(r"\b1sg\b", "first person singular", gloss)
    gloss = re.sub(r"\b2sg\b", "second person singular", gloss)

    # Remove punctuation-only parentheses (e.g., "(sg)")
    gloss = re.sub(r"\([^a-zA-Z]*\)", "", gloss)

    # Remove leading/trailing punctuation
    gloss = gloss.strip(" .;:,-")

    return gloss



In [88]:
def extract_grammatical_info(gloss):
    info = {}
    if "(sg" in gloss:
        info["Number"] = "Sing"
    if "(pl" in gloss:
        info["Number"] = "Plur"
    if "1sg" in gloss:
        info["Person"] = "1"
        info["Possessive"] = True
    if "2sg" in gloss:
        info["Person"] = "2"
    if "3sg" in gloss:
        info["Person"] = "3"
    if "incl" in gloss:
        info["Clusivity"] = "Inclusive"
    if "excl" in gloss:
        info["Clusivity"] = "Exclusive"
    return info


In [91]:
for entry, data in tqdm(dictionary_with_metadata.items()):
    for sense in data["senses"]:
        translation = sense["translation"]
        if not translation or not translation.strip():
            continue
        
        # --- Clean and extract grammatical info first ---
        gloss_features = extract_grammatical_info(translation)
        cleaned_translation = clean_gloss_text(translation)
        sense["cleaned_translation"] = cleaned_translation
        sense["grammatical_features"] = gloss_features
        # -------------------------------------------------------

        if not cleaned_translation.strip():
            continue

        doc = nlp(cleaned_translation)
        multiword = False

        #print(len(doc))
        if len(doc)>1:
            multiword = True
        if len(doc)==0:
            continue

        head = [token for token in doc if token.head == token][0]  # root token
        
        #print(doc[0].text, doc[0].lemma_, doc[0].pos_, doc[0].tag_, doc[0].morph)
        result_dict = {
            "is_multiword": multiword,
            "pos_sequence": [token.pos_ for token in doc],
            "tag_sequence": [token.tag_ for token in doc],
            "morph_sequence": [token.morph.to_dict() for token in doc],
            "lemmas": [token.lemma_ for token in doc],
            "text": [token.text for token in doc],
            "head_index": head.i,
            "head_text": head.text,
            "head_pos": head.pos_,
            "head_lemma": head.lemma_,
            "dependencies": [{"token": t.text, "dep": t.dep_, "head": t.head.text} for t in doc]
        }
        #print(result_dict)
        sense["spacy_analysis"] = result_dict



100%|██████████| 2774/2774 [01:04<00:00, 43.18it/s]


In [92]:
# Store the metadata dictionary
lang_name = "kalamang"
with open(f"dictionary/{lang_name}_dictionary_with_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dictionary_with_metadata, f, ensure_ascii=False, indent=2)

# Dictionary metadata extraction

In [ ]:
import json

with open("dictionary/kalamang_dictionary.json","r",encoding="utf-8") as f:
    dictionary = json.load(f)



In [ ]:
# Clean dictionary (remove entries with empty keys or values)
dictionary = {k: v for k, v in dictionary.items() if k and v and len(v)>0}

dictionary

{'=a': 'focus marker',
 'a': 'filler',
 'a’a': 'yes',
 'adat': 'tradition',
 'ade': 'pejorative interjection',
 'adi': 'interjection of pain',
 'adu': 'interjection of surprise or pain',
 'afukat': 'avocado',
 'ahat': 'sunday',
 '-ahutak': 'alone',
 'ajar': 'v teach () v continue ; continue ; teach',
 'ak': 'sea-side',
 'akal': 'sense',
 'aknar': 'chest',
 'aknar kangun': 'collar bone',
 'akpis': 'convex side',
 '*al': 'string',
 'alangan': 'unable to do',
 'alanganrep': 'look for trouble',
 'alar': 'fish ; k.o. fish',
 'alkon': 'one string',
 'Almahera': 'Halmahera',
 'am': 'breast',
 'am belun': 'nipple',
 'am perun': 'breast milk',
 'amdir': 'garden',
 'amdir komaruk': 'clear land',
 'amkeit': 'give birth',
 'an': 'I',
 '-an': 'my',
 'anahutak': 'I alone',
 'andain': 'I alone',
 'Andan': 'Banda Islands',
 'ang': 'turban shell',
 'anggas': 'door',
 'anggas padenun': 'doorpost',
 'anggon': 'my',
 'anka': 'number',
 'anti': 'antidote; resistant',
 'anting': 'earrings',
 'ao': 'interjec

In [ ]:
# Read kalamng_dictionary_eng.json

import json
with open("dictionary/kalamang_dictionary_eng.json", "r", encoding="utf-8") as f:
    existing_dict = json.load(f)

# Reverse the existing dictionary
reversed_existing_dict = {v: k for k, v in existing_dict.items()}

# Read kalamang_dictionary.json
with open("dictionary/kalamang_dictionary.json", "r", encoding="utf-8") as f:
    existing_dict_kalamang = json.load(f)
# Merge dictionaries, giving priority to the newly extracted entries
merged_dict = {**reversed_existing_dict, **existing_dict_kalamang}

merged_dict

In [ ]:
print(len(existing_dict_kalamang))
print(len(reversed_existing_dict))

In [ ]:
len(merged_dict)

In [ ]:
dict_folder = "dictionary"
import os
os.makedirs(dict_folder, exist_ok=True)
with open(os.path.join(dict_folder, "kalamang_dictionary.json"), "w", encoding="utf-8") as f:
    json.dump(merged_dict, f, ensure_ascii=False, indent=2)

In [ ]:
dict_folder = "dictionary"
if not os.path.exists(dict_folder):
    os.makedirs(dict_folder)

with open(os.path.join(dict_folder, "kalamang_dictionary.json"), "w", encoding="utf-8") as f:
    json.dump(dictionary, f, ensure_ascii=False, indent=4)

In [ ]:
# Remove pronunciation betwee/ /
for key in list(dictionary.keys()):
    cleaned_key = re.sub(r'/[^/]+/', '', key).strip()
    cleaned_value = re.sub(r'/[^/]+/', '', dictionary[key]).strip()
    if cleaned_key != key or cleaned_value != dictionary[key]:
        del dictionary[key]
        dictionary[cleaned_key] = cleaned_value

dictionary

with open(os.path.join(dict_folder, "sursilvan_romansh_dictionary_cleaned.json"), "w", encoding="utf-8") as f:
    json.dump(dictionary, f, ensure_ascii=False, indent=4)

In [ ]:
text_sizes_set = set(text_sizes)
print("Unique text sizes found:", len(text_sizes_set))
print("Text sizes:", sorted(text_sizes_set))
print("Total text sizes collected:", len(text_sizes))

counter = {}
for size in text_sizes_set:
    counter[size] = text_sizes.count(size)

for size, count in sorted(counter.items(), key=lambda x: x[0]):
    print(f"Size: {size}, Count: {count}")

In [ ]:
# Size font tuples counter
size_font_counter = {}
for size, font in size_font_tuples:
    if (size, font) not in size_font_counter:
        size_font_counter[(size, font)] = 0
    size_font_counter[(size, font)] += 1

    

# Get max count
max_count = max(counter.values())
max_size_font = max(size_font_counter.items(), key=lambda x: x[1])[0]
print(f"Max count: {max_count}, Size: {max_size_font[0]}, Font: {max_size_font[1]}")
print("---")
for (size, font), count in sorted(size_font_counter.items(), key=lambda x: x[0]):
    print(f"Size: {size}, Font: {font}, Count: {count}")

In [ ]:
# Draw histogram of text sizes
import matplotlib.pyplot as plt
plt.hist(text_sizes, bins=50, color='blue', alpha=0.7)
plt.title('Histogram of Text Sizes')
plt.xlabel('Text Size')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()


In [ ]:
def collect_consecutive_lines_across_blocks(start_block_index, start_line_index, blocks, target_font=None):
    collected_text = []
    block_index = start_block_index
    line_index = start_line_index

    if start_block_index >= len(blocks):
        return block_index, line_index, "", target_font

    while block_index < len(blocks):
        lines = blocks[block_index].get("lines", [])
        while line_index < len(lines):
            spans = lines[line_index].get("spans", [])
            if not spans or len(spans) != 1:
                return block_index, line_index, " ".join(collected_text), target_font

            span = spans[0]
            if target_font is None:
                target_font = span["font"]

            if span["font"] != target_font:
                return block_index, line_index, " ".join(collected_text), target_font

            collected_text.append(span["text"].strip())
            line_index += 1

        block_index += 1
        line_index = 0

    return block_index, line_index, " ".join(collected_text), target_font


In [ ]:
# Collect all consecutive lines with the same font across blocks
collected_text_sizes = []
sizes_text_lengths = {}
for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    for block_index, block in enumerate(blocks):
        lines = block.get("lines", [])
        for line_index, line in enumerate(lines):
            _, _, collected_text, target_font = collect_consecutive_lines_across_blocks(
                block_index, line_index, blocks
            )
            if collected_text:
                for span in line.get("spans", []):
                    collected_text_sizes.append(span["size"])
                    if sizes_text_lengths.get(span["size"]) is None:
                        sizes_text_lengths[span["size"]] = len(collected_text)
                    else:
                        sizes_text_lengths[span["size"]] += len(collected_text)
                    print(f"Collected text: {collected_text}, Font: {target_font}, Size: {span['size']}")

print("Collected text sizes across blocks:", collected_text_sizes)
# No of blocks
print("Number of blocks:", len(collected_text_sizes))


In [ ]:
sizes_text_lengths_sorted = sorted(sizes_text_lengths.items(), key=lambda x: x[1], reverse=True)
sizes_text_lengths_sorted

In [ ]:
# Take font size with max content and consider it as main body font size. Now find the font size which appears 
main_body_font_size = sizes_text_lengths_sorted[0][0]
print("Main body font size:", main_body_font_size)

# Near body fonts
near_body_fonts = [size for size in sizes_text_lengths_sorted if abs(size[0] - main_body_font_size) <= 1]
print("Near body font sizes:", near_body_fonts)
minimum_near_body_font_size = min(near_body_fonts, key=lambda x: x[1])
maximum_near_body_font_size = max(near_body_fonts, key=lambda x: x[1])
print("Maximum near body font size:", maximum_near_body_font_size)
print("Minimum near body font size:", minimum_near_body_font_size)

# Smaller than minimum near body font size
smaller_than_near_body = [size for size in sizes_text_lengths_sorted if size[0] < minimum_near_body_font_size[0]]
print("Smaller than minimum near body font sizes:", smaller_than_near_body)

# Larger than maximum near body font size
larger_than_near_body = [size for size in sizes_text_lengths_sorted if size[0] > maximum_near_body_font_size[0]]
print("Larger than maximum near body font sizes:", larger_than_near_body)

In [ ]:
# Take larger than maximum near body font size as headings, get number of lines from counter variable before calculated

# TODO: These thresholds are arbitrary, adjust them later
length_threshold = 30
lines_threshold = 5

headings_filtered = []

for size,length in larger_than_near_body:
    no_lines = counter.get(size, -1)
    print(f"Size: {size}, Length: {length}, No. of lines: {no_lines}")
    if length > length_threshold and no_lines > lines_threshold:
        headings_filtered.append((size, length, no_lines))

headings_filtered



In [ ]:
# sort by size and mark as heading, subheading and sub-subheading
headings_sorted = sorted(headings_filtered, key=lambda x: x[0], reverse=True)

headings_sizes = [size for size, length, no_lines in headings_sorted]

headings_sizes
    


In [ ]:
for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    for block_index, block in enumerate(blocks):
        lines = block.get("lines", [])
        for line_index, line in enumerate(lines):
            spans = line.get("spans", [])
            for span in spans:
                text = span["text"].strip()
                size = span["size"]
                font = span["font"]
                if size in headings_sizes:
                    heading_index = headings_sizes.index(size) + 1
                    print(f"Heading found: {text}, Font: {font}, H{heading_index}")


In [ ]:
import re

# Parse through and collect sections based on headings
section_text = []  # List of sections
current_section = None
parallel_sentences = []

parallel_temp_storage = []
parallel_y_pos = None
ilg_temp_storage = []

text_buffer = []  # Buffer to collect text for the current section

parallel_flag = False

for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    for block_index, block in enumerate(blocks):
        lines = block.get("lines", [])
        for line_index, line in enumerate(lines):
            spans = line.get("spans", [])
            for span in spans:
                text = span["text"].strip()
                size = span["size"]
                font = span["font"]
                # y position
                x_pos,y_pos = span["bbox"][:2]
                if not text:
                    continue

                #print(text, size, font)

                

                if size in headings_sizes:
                    heading_index = headings_sizes.index(size) + 1
                    # store the current section if it exists
                    if current_section is not None:
                        section_text.append(current_section)
                    current_section = {
                        "heading": text,
                        "text": "",
                        "header_number": heading_index,
                        "font_style": font,
                        "font_size": size,
                        "page_number": page_num + 1,
                    }
                    #print(f"Heading found: {text}, Font: {font}, H{heading_index}")
                elif current_section is not None and size == main_body_font_size:
                    # Check for parallel sentences (starts with a paranthesis with number inside)
                    # TODO: HARDCODED TEMPORARY FIX, GENERALIZE LATER
                    if font == "LibertinusSerif-Italic":
                        if parallel_flag == False:
                            parallel_flag = True
                            parallel_temp_storage.append(text)
                            parallel_y_pos = y_pos
                            text_buffer.append(text)
                        else:
                            if parallel_y_pos is not None and y_pos == parallel_y_pos:
                                parallel_temp_storage.append(text)
                                text_buffer.append(text)
                            else:
                                parallel_flag = False
                                parallel_y_pos = None
                    elif parallel_flag == True and len(text.split(" ")) == 1 and y_pos > parallel_y_pos:
                        

                        ilg_temp_storage.append(text)
                        text_buffer.append(text)
                        print(text,size,font,x_pos,y_pos)
                    else:
                        parallel_flag = False
                        parallel_y_pos = None
                    if re.match(r'^\(\d+\)', text):
                        parallel_sentences.append(text)
                        #print(text,size,font,x_pos,y_pos)
                        continue
                    if font == max_size_font[1]:
                        if current_section["text"]:
                            current_section["text"] += " "
                        current_section["text"] += text



if current_section is not None:
    section_text.append(current_section)

In [ ]:
# print how many sections were found
print(f"Total sections found: {len(section_text)}")

In [ ]:
for section in section_text:
    print(f"{section['heading']}, Header Number: {section['header_number']}, Font: {section['font_style']}, Size: {section['font_size']}, Page: {section['page_number']}")
    print(f"  {section['text'][:500]}...")  # Print first 100 characters of text
    print("---")

In [ ]:
import json

# Save as json file

with open("parsed_grammar_json/sursilvan_romansh_sections.json", "w", encoding="utf-8") as f:
    json.dump(section_text, f, ensure_ascii=False, indent=4)

# Parallel sentences parsing

In [ ]:
all_lines = []

# First collect all text as lines instead of blocks
for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    for block in blocks:
        lines = block.get("lines", [])
        for line_index, line in enumerate(lines):
            spans = line.get("spans", [])

            for span_index, span in enumerate(spans):
                text = span["text"].strip()
                size = span["size"]
                font = span["font"]
                x_pos,y_pos = span["bbox"][:2]
                if not text:
                    continue
                all_lines.append({
                    "text": text,
                    "size": size,
                    "font": font,
                    "page_number": page_num + 1,
                    "x_position": x_pos,
                    "y_position": y_pos
                })




In [ ]:
all_lines

In [ ]:
from langdetect import detect_langs
detect_langs("This is a test sentence.")  # → 'en'


In [ ]:
import fasttext

# Load model (adjust path if you saved it elsewhere)
model = fasttext.load_model("lid.176.bin")

def detect_language_fasttext(text):
    # normalize common curly quotes
    clean = text.replace("’", "'").replace("“", '"').replace("”", '"')
    
    labels, probs = model.predict(clean, k=1)  # top-1 prediction
    lang = labels[0].replace("__label__", "")
    prob = float(probs[0])
    return lang, prob


In [ ]:


def classify_text_igt_english(text):
    if text is None or len(text.strip()) == 0:
        return "reject"
    IGT_ABBRS = {
        "1","2","3","SG","1SG","2SG","3SG","PL","DU","PST","PRS","FUT","IPFV","PFV","PROG","HAB",
        "NOM","ACC","ERG","GEN","DAT","LOC","ALL","ABL","INS","COM","BEN","TOP","FOC","LAT",
    "CAUS","PASS","MID","REFL","RECIP","AUX","COP","NEG","COND","IMP","SUBJ","IND","POSS","PROX"
    }

    IGT_ABBRS = {abbr.lower() for abbr in IGT_ABBRS}

    

    gloss_like_score = 0
    normalized_english_score = 0
    words = text.split()
    total_words = len(words)
    print(words)
    for word in words:

        subwords = re.split(r'[.\-=]', word)

        for sw in subwords:
            if sw in IGT_ABBRS:
                gloss_like_score += 2
                print(sw)
            # else if remove all numbers from subword, if the remaining part is in IGT_ABBRS, add 1 to gloss score
            else:
                sw_no_num = re.sub(r'\d+', '', sw)
                if sw_no_num in IGT_ABBRS:
                    gloss_like_score += 1
                    print(sw_no_num)
    
    try:
        # clean text (remove any text within brackets and punctuation)
        res = re.sub(r'\[.*?\]', '', text)
        res = re.sub(r'[^\w\s]', '', res)    
        res = res.strip().lower()
        res = detect_language_fasttext(res)
        if res and res[0] == 'en':
            normalized_english_score = res[1]
    except Exception as e:
        #print(f"Error detecting language: {e}")
        #print("Text causing error:", text)
        return "reject"

    normalized_gloss_score = gloss_like_score / total_words

    print(f"Text: {text}, Gloss score: {normalized_gloss_score}, English score: {normalized_english_score}, Total words: {total_words}")

    if normalized_gloss_score > 0.1:
        return "IGT"
    elif normalized_english_score > 0.8:
        return "English"
    else:
        return "unknown"

In [ ]:
parallel_sentences = []

idx = 0
while idx < len(all_lines):
    line = all_lines[idx]
    text = line["text"]
    page_number = line["page_number"]
    if re.fullmatch(r'\(\d+\)', text):
        print(f"Parallel sentence found at page {page_number}: {text}")
        top_x = None
        bottom_y = None
        new_line = all_lines[idx+1]
        new_text = new_line["text"]
        new_x_pos = new_line["x_position"]
        top_x = new_x_pos

        #print(top_x)
        temp = []
        translation_idx = None
        counter = 0
        while idx < len(all_lines)-1:
            idx+=1
            new_line = all_lines[idx]
            new_text = new_line["text"]
            new_x_pos,new_y_pos = new_line["x_position"],new_line["y_position"]
            if(top_x - new_x_pos > 2):
                break
                temp.append((new_text,new_x_pos,new_y_pos))
            temp.append((new_text,new_x_pos,new_y_pos))
        #print(temp)
        # Sort temp by y position
        temp_sorted = sorted(temp, key=lambda x: x[2])
        
        # join all text in temp_sorted which are in same y position
        joined_text = []
        current_y = None
        current_line = []
        for t in temp_sorted:
            if current_y is None:
                current_y = t[2]
                current_line.append(t[0])
            elif abs(t[2] - current_y) < 2:
                current_line.append(t[0])
            else:
                joined_text.append(" ".join(current_line))
                current_y = t[2]
                current_line = [t[0]]

        if current_line:
            joined_text.append(" ".join(current_line))

        # Classify each line in the joined text as IGT, english or other based on some heuristics
        igt = []
        english = []
        other = []

        for i,line in enumerate(joined_text):
            result = classify_text_igt_english(line)
            if result == "IGT":
                igt.append(line)
            elif result == "English":
                english.append(line)
                # For now, fix later TODO
                # append rest of lines to english as well
                english.extend(joined_text[i+1:])
                break
            elif result == "reject":
                continue
            else:
                other.append(line)

        igt = " ".join(igt)
        english = " ".join(english)
        other = " ".join(other)
        
        print("text:", other)
        print("IGT:", igt)
        print("English:", english)
        print("---")
        #check lengths of the texts, if too long, it's a false flag
        if len(igt) > 1000 or len(english) > 1000 or len(other) > 1000:
            print("False flag, skipping...")
            continue

        # Clean parallel sentences
        if (len(other) < 1 or len(english) < 1 ):
            print("Too short, skipping...")
            continue
        parallel_sentences.append({
            "page_number": page_number,
            "text": other,
            "IGT": igt,
            "English": english,
        })
    else:
        idx+=1


In [ ]:
parallel_sentences

In [ ]:
# store parallel sentences
with open("parallel_sents/sursilvan_romansh_parallel_sentences.json", "w", encoding="utf-8") as f:
    json.dump(parallel_sentences, f, ensure_ascii=False, indent=4)

In [ ]:
import fasttext

# Load model (adjust path if you saved it elsewhere)
model = fasttext.load_model("lid.176.bin")

def detect_language(text):
    # normalize common curly quotes
    clean = text.replace("’", "'").replace("“", '"').replace("”", '"')
    
    labels, probs = model.predict(clean, k=1)  # top-1 prediction
    lang = labels[0].replace("__label__", "")
    prob = float(probs[0])
    return lang, prob

print(detect_language("It’s like water."))
print(detect_language("C’est comme de l’eau."))
print(detect_language("È come l’acqua."))
